# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdul-ITexpert/flyrank-internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Rule

I will create a simple baseline score for content pages using two signals: **content staleness** and **search performance**.

Pages that are more stale and show stronger evidence of search opportunity will receive a higher score. The score will rank pages for content review and support a `REFRESH` or `MONITOR` action.

This is a simple decision-support rule, not a prediction that a refresh will definitely improve performance.

### Reason codes

- `STALE_SEARCH_OPPORTUNITY` — the page is stale and also shows a search opportunity.
- `STALE_CONTENT` — the page is stale but does not show a strong search opportunity.
- `SEARCH_OPPORTUNITY` — the page shows a search opportunity but is not strongly stale.
- `MONITOR` — the page does not have enough evidence for a refresh recommendation.

### Actions

- `REFRESH` — review the page and consider updating the content.
- `MONITOR` — keep the page under observation instead of prioritizing a refresh.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)
content_columns = con.sql("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
""").df()

display(content_columns)


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [10]:
signal1 = con.sql("""
WITH content AS (
    SELECT
        content_hash_id,
        content_updated_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
),

march AS (
    SELECT
        d.content_hash_id,
        AVG(d.gsc_impressions) AS avg_impressions,
        AVG(d.gsc_clicks) AS avg_clicks,
        MAX(d.report_date) AS report_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    ) d
    WHERE d.month = '2026-03'
      AND d.gsc_data_available IS TRUE
    GROUP BY d.content_hash_id
)

SELECT
    CASE
        WHEN date_diff('day', c.content_updated_date, m.report_date) <= 30
            THEN '0-30 days'
        WHEN date_diff('day', c.content_updated_date, m.report_date) <= 90
            THEN '31-90 days'
        WHEN date_diff('day', c.content_updated_date, m.report_date) <= 180
            THEN '91-180 days'
        ELSE '180+ days'
    END AS staleness_bucket,

    COUNT(*) AS n,
    AVG(m.avg_impressions) AS avg_impressions,
    AVG(m.avg_clicks) AS avg_clicks

FROM march m
JOIN content c
    ON m.content_hash_id = c.content_hash_id

WHERE c.content_updated_date IS NOT NULL

GROUP BY staleness_bucket
ORDER BY
    CASE staleness_bucket
        WHEN '0-30 days' THEN 1
        WHEN '31-90 days' THEN 2
        WHEN '91-180 days' THEN 3
        WHEN '180+ days' THEN 4
    END
""").df()

display(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_impressions,avg_clicks
0,0-30 days,152471,55.747853,0.173715
1,31-90 days,22683,48.608831,0.109816
2,91-180 days,1373,12.397079,0.052092
3,180+ days,211,4.671952,0.009077


### Signal 1: Staleness — Verdict: CONFIRMED

I bucketed content by the number of days since its last update. Older content consistently has lower average impressions and clicks. Average clicks fall from 0.174 for content updated within 30 days to 0.009 for content older than 180 days. This supports staleness as a useful signal for prioritizing content for review.

In [7]:
signal2 = con.sql("""
SELECT
    CASE
        WHEN gsc_avg_position <= 10 THEN '1-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        WHEN gsc_avg_position <= 50 THEN '21-50'
        ELSE '50+'
    END AS position_bucket,
    COUNT(*) AS n,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY position_bucket
ORDER BY position_bucket
""").df()

display(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_impressions,avg_clicks
0,1-10,2183484,87.868153,0.298278
1,11-20,519223,56.596118,0.178053
2,21-50,631491,88.590989,0.121283
3,50+,276863,12.527727,0.005450


### Signal 2: Search performance — Verdict: MIXED

I bucketed pages by average Google Search Console position. Pages in positions 50+ have much lower average impressions and clicks, while positions 1–10 have the strongest average clicks. However, the relationship is not perfectly monotonic because the 21–50 bucket has slightly higher average impressions than the 11–20 bucket. Therefore, I mark this signal as MIXED rather than CONFIRMED.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

march_df = con.sql("""
WITH content AS (
    SELECT
        content_hash_id,
        content_updated_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
),

performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
)

SELECT
    p.client_hash_id,
    p.content_hash_id,
    p.report_date,
    c.content_updated_date,
    date_diff('day', c.content_updated_date, p.report_date) AS days_stale,
    p.gsc_impressions,
    p.gsc_clicks,
    p.gsc_avg_position

FROM performance p
LEFT JOIN content c
    ON p.content_hash_id = c.content_hash_id
WHERE c.content_updated_date IS NOT NULL
""").df()

display(march_df.head())
print("Rows:", len(march_df))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,content_updated_date,days_stale,gsc_impressions,gsc_clicks,gsc_avg_position
0,client_2094c6eb080311d5,content_14a3d47ccd0d15dc,2026-03-09,2026-05-12,-64,1,0,5.000000
1,client_2094c6eb080311d5,content_14a6f92117604fef,2026-03-31,2026-05-20,-50,2,0,2.500000
2,client_2094c6eb080311d5,content_14a86c63a214f648,2026-03-21,2026-05-12,-52,1,0,1.000000
3,client_2094c6eb080311d5,content_14b1a02c1b8557fb,2026-03-28,2026-05-12,-45,7,0,42.857143
4,client_2094c6eb080311d5,content_14c17f59aa610ab3,2026-03-31,2026-06-22,-83,9,0,2.666667


Rows: 3611061


In [12]:
import numpy as np

df = march_df.copy()

# Staleness score
df["staleness_score"] = np.select(
    [
        df["days_stale"] > 180,
        df["days_stale"] > 90,
        df["days_stale"] > 30
    ],
    [
        3,
        2,
        1
    ],
    default=0
)

# Search opportunity score
df["search_score"] = np.where(
    (df["gsc_avg_position"] > 10) &
    (df["gsc_avg_position"] <= 50),
    1,
    0
)

# Final baseline score
df["score"] = df["staleness_score"] + df["search_score"]

df[[
    "days_stale",
    "gsc_avg_position",
    "staleness_score",
    "search_score",
    "score"
]].head()

,days_stale,gsc_avg_position,staleness_score,search_score,score
0,-64,5.000000,0,0,0
1,-50,2.500000,0,0,0
2,-52,1.000000,0,0,0
3,-45,42.857143,0,1,1
4,-83,2.666667,0,0,0


In [13]:
df["reason_code"] = np.select(
    [
        (df["staleness_score"] >= 2) & (df["search_score"] == 1),
        (df["staleness_score"] >= 2),
        (df["search_score"] == 1)
    ],
    [
        "STALE_SEARCH_OPPORTUNITY",
        "STALE_CONTENT",
        "SEARCH_OPPORTUNITY"
    ],
    default="MONITOR"
)

In [14]:
df["action"] = np.where(
    df["score"] >= 2,
    "REFRESH",
    "MONITOR"
)

In [15]:
queue = df.sort_values(
    ["score", "days_stale"],
    ascending=[False, False]
).copy()

queue[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "score",
        "reason_code",
        "action"
    ]
].head(10)

,client_hash_id,content_hash_id,report_date,score,reason_code,action
1189117,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-31,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1188827,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-30,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190261,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-29,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1193519,client_65de48885f4ef01b,content_38c60323fd1608ec,2026-03-29,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1189382,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-28,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1189626,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-27,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1189850,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-26,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190061,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-25,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190452,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-24,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190634,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-23,4,STALE_SEARCH_OPPORTUNITY,REFRESH


In [16]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")

Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "days_stale",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action"
    ]
]

,client_hash_id,content_hash_id,report_date,days_stale,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action
1189117,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-31,303,1,0,48.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1188827,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-30,302,1,0,50.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190261,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-29,301,1,0,47.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1193519,client_65de48885f4ef01b,content_38c60323fd1608ec,2026-03-29,301,2,0,12.500000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1189382,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-28,300,1,0,48.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1189626,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-27,299,1,0,48.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1189850,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-26,298,1,0,49.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190061,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-25,297,1,0,46.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190452,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-24,296,1,0,46.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH
1190634,client_65de48885f4ef01b,content_19f71daba0876547,2026-03-23,295,1,0,47.000000,4,STALE_SEARCH_OPPORTUNITY,REFRESH


In [18]:
top20["confidence"] = "MEDIUM"
top20["review_reason"] = (
    "Selected because the baseline score prioritizes "
    "staleness and search-position opportunity."
)

top20["what_would_make_it_wrong"] = (
    "The page may be intentionally old, seasonal, or already "
    "performing adequately despite the rule's signals."
)

In [19]:
top20[
    [
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "confidence",
        "review_reason",
        "what_would_make_it_wrong"
    ]
]

,content_hash_id,score,reason_code,action,confidence,review_reason,what_would_make_it_wrong
1189117,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1188827,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1190261,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1193519,content_38c60323fd1608ec,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1189382,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1189626,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1189850,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1190061,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1190452,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."
1190634,content_19f71daba0876547,4,STALE_SEARCH_OPPORTUNITY,REFRESH,MEDIUM,Selected because the baseline score prioritize...,"The page may be intentionally old, seasonal, o..."


## 3. Top-20 review

1. **content_19f71daba0876547** — **REFRESH**. Why: score 4 because the rule identifies both staleness and a search-position opportunity. Confidence: Medium. What could make it wrong: the content may be intentionally old or the search-position signal may not represent a real refresh opportunity.

2. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with `STALE_SEARCH_OPPORTUNITY`. Confidence: Medium. What could make it wrong: the recommendation may be repeated because the same content appears on another report-date/client row.

3. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with both baseline signals triggered. Confidence: Medium. What could make it wrong: the page may not benefit from an update even though the rule flags it.

4. **content_38c60323fd1608ec** — **REFRESH**. Why: score 4 and the rule identifies staleness plus search opportunity. Confidence: Medium. What could make it wrong: the position signal was only MIXED in the signal audit.

5. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with `STALE_SEARCH_OPPORTUNITY`. Confidence: Medium. What could make it wrong: repeated rows may represent the same content rather than independent opportunities.

6. **content_19f71daba0876547** — **REFRESH**. Why: score 4 because both rule signals are active. Confidence: Medium. What could make it wrong: the content could be intentionally maintained without frequent updates.

7. **content_19f71daba0876547** — **REFRESH**. Why: score 4 and the rule flags a stale search opportunity. Confidence: Medium. What could make it wrong: the observed search position may not translate into an actionable content improvement.

8. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with the same reason code. Confidence: Medium. What could make it wrong: this may be another repeated observation of the same content.

9. **content_19f71daba0876547** — **REFRESH**. Why: score 4 and both signals contribute to the recommendation. Confidence: Medium. What could make it wrong: seasonality or intentional evergreen content could make the refresh unnecessary.

10. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with `STALE_SEARCH_OPPORTUNITY`. Confidence: Medium. What could make it wrong: the rule does not know whether the content is strategically important or already satisfactory.

11. **content_ee6f61ff4145746c** — **REFRESH**. Why: score 4 because the rule flags staleness and search opportunity. Confidence: Medium. What could make it wrong: the mixed search-position signal may produce a false opportunity.

12. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with both signals triggered. Confidence: Medium. What could make it wrong: this could be a repeated daily/client observation rather than a separate page opportunity.

13. **content_19f71daba0876547** — **REFRESH**. Why: score 4 and `STALE_SEARCH_OPPORTUNITY`. Confidence: Medium. What could make it wrong: age alone does not prove that updating the content will improve performance.

14. **content_a2e85700106a33b8** — **REFRESH**. Why: score 4 with both staleness and search opportunity signals. Confidence: Medium. What could make it wrong: the page may have low strategic value despite meeting the rule.

15. **content_cdd557d042d8a774** — **REFRESH**. Why: score 4 and the rule identifies a stale search opportunity. Confidence: Medium. What could make it wrong: search-position opportunity is not a guaranteed content-refresh opportunity.

16. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with `STALE_SEARCH_OPPORTUNITY`. Confidence: Medium. What could make it wrong: repeated content observations can make the queue look larger than the number of unique pages.

17. **content_cdd557d042d8a774** — **REFRESH**. Why: score 4 because both signals are active. Confidence: Medium. What could make it wrong: the content may be intentionally old or stable.

18. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with the same reason code. Confidence: Medium. What could make it wrong: the same content appears multiple times in the top queue.

19. **content_cdd557d042d8a774** — **REFRESH**. Why: score 4 and the rule identifies a stale search opportunity. Confidence: Medium. What could make it wrong: the mixed position signal could make this a weak recommendation.

20. **content_19f71daba0876547** — **REFRESH**. Why: score 4 with `STALE_SEARCH_OPPORTUNITY`. Confidence: Medium. What could make it wrong: the rule cannot determine whether a refresh would actually improve future performance.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak_picks = queue.sort_values(
    ["score", "days_stale"],
    ascending=[True, True]
).head(10)

display(
    weak_picks[
        [
            "content_hash_id",
            "days_stale",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "score",
            "reason_code",
            "action"
        ]
    ]
)


,content_hash_id,days_stale,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action
12069,content_33305598fdd2f38c,-127,4,0,5.750000,0,MONITOR,MONITOR
13712,content_337d79022dffea47,-127,8,0,3.500000,0,MONITOR,MONITOR
15557,content_21c4ffde557f0909,-127,48,0,8.354167,0,MONITOR,MONITOR
15614,content_5011cfee8f5e75f5,-127,30,0,6.633333,0,MONITOR,MONITOR
30690,content_732b4284910da21d,-127,15,0,8.333333,0,MONITOR,MONITOR
45864,content_964d3b4233fc16a8,-127,6,0,8.666667,0,MONITOR,MONITOR
46177,content_9fa132b5d23fca8f,-127,197,0,7.568528,0,MONITOR,MONITOR
46433,content_a298a5475bbf4048,-127,6,0,3.333333,0,MONITOR,MONITOR
71103,content_06844acfabb5583a,-127,43,0,1.953488,0,MONITOR,MONITOR
113446,content_0a9b787d28fc695c,-127,386,0,51.756477,0,MONITOR,MONITOR


## 4. Weak picks and limitation

### Weak picks

The weakest picks show an important limitation of the baseline rule. The rule is intentionally simple and relies mainly on content staleness and search position. A high score does not guarantee that refreshing a page will improve performance.

Some rows also represent the same `content_hash_id` multiple times because the underlying daily table has one row per client, content item, and report date. Therefore, the ranked queue can contain repeated observations of the same content item instead of a unique content-level action queue.

### Main limitation

The biggest limitation is that this baseline is a simple heuristic, not a learned model. It does not understand content quality, topic importance, seasonality, or the actual future impact of a refresh. The search-position signal was also MIXED in the signal audit, so it should not be treated as a guaranteed opportunity.

A future version should aggregate to one row per client-content item before ranking and eventually use a future performance label to learn which refreshes actually lead to improvement.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.